# Chagas Drug Discovery | Descritores Moleculares e Preprocessing
**Objetivo:** Calcular 1875 descritores moleculares (2D + 3D) via PaDEL para inibidores e decoys, e construir os datasets finais para modelagem.
**Pipeline:** Lotes DUD-E → amostragem de decoys → cálculo de descritores (`from_smiles`) → datasets finais.

## 0. Imports e configuração

In [1]:
import pandas as pd
import numpy as np
import os
from padelpy import padeldescriptor
import tarfile
import glob
import io
from padelpy import from_smiles
import tqdm
import warnings

os.makedirs('../data/raw/dude_batches', exist_ok=True)
os.makedirs('../data/raw/dude_results', exist_ok=True)
os.makedirs('../data/processed', exist_ok=True)

df_inhibitors = pd.read_csv('../data/raw/cruzipain_final.csv')
print(f"Inibidores carregados: {len(df_inhibitors)}")

Inibidores carregados: 508


## 1. Verificação do dataset de inibidores
Confirma que os 508 inibidores estão sem duplicatas antes de gerar os lotes.

In [2]:
df_inhibitors = pd.read_csv(r'..\data\raw\cruzipain_final.csv')
print(f"Total: {len(df_inhibitors)}")
print(f"Duplicatas: {df_inhibitors['smiles'].duplicated().sum()}")

Total: 508
Duplicatas: 0


## 2. Geração de lotes para o DUD-E
O DUD-E aceita no máximo 50 SMILES por submissão. Os 508 inibidores são divididos em 11 lotes de 50 para geração de decoys em `dude.docking.org/generate`.

In [3]:
BATCH_SIZE = 50
smiles_list = df_inhibitors['smiles'].tolist()
batch_paths = []

for i, start in enumerate(range(0, len(smiles_list), BATCH_SIZE)):
    batch = smiles_list[start:start + BATCH_SIZE]
    path  = f'../data/raw/dude_batches/batch_{i:02d}.smi'
    pd.Series(batch).to_csv(path, index=False, header=False)
    batch_paths.append(path)
    print(f"Lote {i:02d}: {len(batch)} SMILES -> {path}")

print(f"\nTotal: {len(batch_paths)} lotes criados")

Lote 00: 50 SMILES -> ../data/raw/dude_batches/batch_00.smi
Lote 01: 50 SMILES -> ../data/raw/dude_batches/batch_01.smi
Lote 02: 50 SMILES -> ../data/raw/dude_batches/batch_02.smi
Lote 03: 50 SMILES -> ../data/raw/dude_batches/batch_03.smi
Lote 04: 50 SMILES -> ../data/raw/dude_batches/batch_04.smi
Lote 05: 50 SMILES -> ../data/raw/dude_batches/batch_05.smi
Lote 06: 50 SMILES -> ../data/raw/dude_batches/batch_06.smi
Lote 07: 50 SMILES -> ../data/raw/dude_batches/batch_07.smi
Lote 08: 50 SMILES -> ../data/raw/dude_batches/batch_08.smi
Lote 09: 50 SMILES -> ../data/raw/dude_batches/batch_09.smi
Lote 10: 8 SMILES -> ../data/raw/dude_batches/batch_10.smi

Total: 11 lotes criados


## 3. Consolidação dos decoys DUD-E
Lê todos os arquivos `.picked` dentro dos `.tar.gz` retornados pelo DUD-E e consolida em um único dataset de decoys únicos.
Formato dos arquivos: `SMILES TAB compound_id TAB picked_id`, apenas a coluna de SMILES é extraída.

In [4]:
all_decoys = []

for tar_path in sorted(glob.glob('../data/raw/dude_results/*.tar.gz')):
    with tarfile.open(tar_path, 'r:gz') as tar:
        for member in tar.getmembers():
            if 'decoys/' in member.name and member.name.endswith('.picked'):
                f = tar.extractfile(member)
                if f:
                    df_batch = pd.read_csv(
                        io.TextIOWrapper(f),
                        sep='\t',
                        header=0,        # primeira linha é cabeçalho
                        usecols=[0],     # só a coluna de SMILES
                        names=['smiles'],
                        skiprows=0
                    )
                    all_decoys.append(df_batch)

df_decoys = pd.concat(all_decoys, ignore_index=True)
df_decoys = df_decoys.drop_duplicates(subset='smiles')
df_decoys['molecule_chembl_id'] = [f'DECOY_{i:04d}' for i in range(len(df_decoys))]
df_decoys['label'] = 0

df_decoys['smiles'].to_csv('../data/raw/decoys_all.smi',
                            index=False, header=False)
print(f"Total de decoys únicos: {len(df_decoys)}")

Total de decoys únicos: 27131


## 4. Amostragem de decoys
O DUD-E gerou 27.131 decoys para 508 inibidores (ratio 53:1). Para evitar desbalanceamento severo no Modelo 1, são amostrados 600 decoys com `random_state=42`, mantendo ratio ~1.2:1.

In [ ]:
N_DECOYS = 600 

df_decoys_sampled = df_decoys.sample(n=N_DECOYS, random_state=42)
df_decoys_sampled = df_decoys_sampled.reset_index(drop=True)

# Salvar arquivo final
df_decoys_sampled['smiles'].to_csv('../data/raw/decoys_all.smi',
                                    index=False, header=False)

print(f"Decoys selecionados: {len(df_decoys_sampled)}")
print(f"Inibidores: {len(df_inhibitors)}")
print(f"Ratio final: {len(df_decoys_sampled)/len(df_inhibitors):.1f}x")

Decoys selecionados: 600
Inibidores: 508
Ratio final: 1.2x


## 5. Geração dos arquivos .smi
Exporta SMILES dos inibidores e decoys em formato `.smi` para uso no PaDEL e rastreabilidade do pipeline.

In [6]:
# Inibidores
df_inhibitors[['smiles', 'molecule_chembl_id']].to_csv(
    '../data/processed/inhibitors.smi',
    sep='\t', index=False, header=False
)

# Decoys
df_decoys_sampled[['smiles', 'molecule_chembl_id']].to_csv(
    '../data/processed/decoys.smi',
    sep='\t', index=False, header=False
)

print(f"inhibitors.smi: {len(df_inhibitors)} moléculas")
print(f"decoys.smi: {len(df_decoys_sampled)} moléculas")

inhibitors.smi: 508 moléculas
decoys.smi: 600 moléculas


In [7]:
# Inibidores — só SMILES
df_inhibitors['smiles'].to_csv(
    r'..\data\processed\inhibitors.smi',
    index=False, header=False
)

# Decoys — só SMILES
df_decoys_sampled['smiles'].to_csv(
    r'..\data\processed\decoys.smi',
    index=False, header=False
)

## 6. Cálculo de descritores moleculares
Utiliza `from_smiles` do padelpy com para calcular 1875 descritores por molécula.
O `timeout=120` garante que moléculas complexas que não convergem na otimização 3D sejam ignoradas em vez de travar o processo.
Moléculas com erro retornam linha de NaN 

In [8]:
df_inhibitors = pd.read_csv(r'..\data\processed\inhibitors.smi', header=None, names=['smiles'])
smiles_list = df_inhibitors['smiles'].tolist()

all_descriptors = []
erros = 0

for smi in tqdm.tqdm(smiles_list):
    try:
        desc = from_smiles(smi, timeout=120)
        all_descriptors.append(desc)
    except Exception as e:
        erros += 1
        all_descriptors.append({})

desc_inh = pd.DataFrame(all_descriptors)
desc_inh.to_csv(r'..\data\processed\descriptors_inhibitors.csv', index=False)
print(f"Cálculo dos descritores dos inibidores concluído. Shape: {desc_inh.shape}")
print(f"Erros: {erros}/{len(smiles_list)}")

df_decoys = pd.read_csv(r'..\data\processed\decoys.smi', header=None, names=['smiles'])
smiles_decoys = df_decoys['smiles'].tolist()

all_descriptors_dec = []
erros_dec = 0

for smi in tqdm.tqdm(smiles_decoys):
    try:
        desc = from_smiles(smi, timeout=120)
        all_descriptors_dec.append(desc)
    except Exception as e:
        erros_dec += 1
        all_descriptors_dec.append({})

desc_dec = pd.DataFrame(all_descriptors_dec)
desc_dec.to_csv(r'..\data\processed\descriptors_decoys.csv', index=False)
print(f"Cálculo dos descritores dos decoys concluído. Shape: {desc_dec.shape}")
print(f"Erros: {erros_dec}/{len(smiles_decoys)}")

100%|██████████| 508/508 [28:39<00:00,  3.38s/it]


Cálculo dos descritores dos inibidores concluído. Shape: (508, 1875)
Erros: 0/508


100%|██████████| 600/600 [43:01<00:00,  4.30s/it]    


Cálculo dos descritores dos decoys concluído. Shape: (600, 1875)
Erros: 1/600


## 8. Construção dos datasets finais
Salva dois arquivos separados:
- `inhibitors_full.csv` | inibidores com SMILES, IC50, potency_class e 1875 descritores
- `decoys_full.csv` | decoys com SMILES e 1875 descritores

Essa separação facilita o uso nos 3 modelos: o Modelo 1 usa os dois, o Modelo 2 e 3 usam apenas inibidores.

In [9]:
df_inhibitors = pd.read_csv(r'..\data\raw\cruzipain_final.csv')

desc_inh = pd.read_csv(r'..\data\processed\descriptors_inhibitors.csv')
desc_dec = pd.read_csv(r'..\data\processed\descriptors_decoys.csv')

# Inibidores — smiles + IC50 + potency_class + descritores
df_inh_final = pd.concat([
    df_inhibitors[['smiles', 'IC50_nM', 'potency_class']].reset_index(drop=True),
    desc_inh.reset_index(drop=True)
], axis=1)

# Decoys — smiles + descritores
df_decoys_smi = pd.read_csv(r'..\data\processed\decoys.smi', header=None, names=['smiles'])
df_dec_final = pd.concat([
    df_decoys_smi.reset_index(drop=True),
    desc_dec.reset_index(drop=True)
], axis=1)

df_inh_final.to_csv(r'..\data\processed\inhibitors_full.csv', index=False)
df_dec_final.to_csv(r'..\data\processed\decoys_full.csv', index=False)

print(f"inhibitors_full.csv: {df_inh_final.shape}")
print(f"decoys_full.csv:     {df_dec_final.shape}")

inhibitors_full.csv: (508, 1878)
decoys_full.csv:     (600, 1876)


## 11. Decisões 

**Descritores:**
- 1875 descritores calculados com `from_smiles` (PaDEL), o padeldescriptor apresentou erros
- 0 erros nos inibidores | 1 erro nos decoys 

**Decoys:**
- 27.131 decoys gerados pelo DUD-E a partir de 11 lotes de 50 SMILES
- 600 amostrados com `random_state=42` para reprodutibilidade

**Datasets finais:**
- `inhibitors_full.csv`: 508 inibidores com SMILES, IC50, potency_class e descritores
- `decoys_full.csv`: 600 decoys com SMILES e descritores
